# Module 7 · Testing AI Applications: From Unit Tests to Behavioral Validation

**Track:** CI/CD & Automation  
**Toolchain:** pytest · unittest · mocking · hypothesis  
**Objective:** Learn how to test AI applications reliably despite their non-deterministic nature.

---

## 🧠 Why Is Testing AI Applications Hard?

Traditional software: `add(2, 3)` → ALWAYS returns `5`. Easy to test.

AI applications: `summarize(article)` → Returns DIFFERENT text every time. How do you test that?

| Traditional Testing | AI Testing |
|--------------------|-----------|
| Exact output matching | Approximate output matching |
| Deterministic | Non-deterministic (same input → different output) |
| Fast (ms) | Slow (seconds for LLM calls) |
| Free | Expensive (API costs per test run) |
| One failure mode | Multiple failure modes (model, data, prompt, infra) |

### The AI Testing Pyramid

```
        [E2E Tests]           ← Full pipeline tests (slow, expensive, few)
       /           \
      [Integration Tests]     ← Component interaction tests (medium)
     /               \
    [Unit Tests]              ← Individual function tests (fast, cheap, many)
   /                   \
  [Contract Tests]            ← Input/output schema validation
```

**Rule:** Most tests should be at the bottom (fast, cheap). Only a few at the top (slow, expensive).

---

## 📑 Table of Contents

* [🧠 Why Is Testing AI Applications Hard?](#why-is-testing-ai-applications-hard)
  * [The AI Testing Pyramid](#the-ai-testing-pyramid)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · Unit Testing AI Components](#1-unit-testing-ai-components)
  * [🤔 What CAN Be Unit Tested in AI?](#what-can-be-unit-tested-in-ai)
  * [The #1 Pattern: Mock the LLM](#the-1-pattern-mock-the-llm)
* [2 · Contract Testing: Schema Validation](#2-contract-testing-schema-validation)
  * [🤔 What is a Contract Test?](#what-is-a-contract-test)
* [3 · Snapshot Testing for Prompts](#3-snapshot-testing-for-prompts)
  * [🤔 What is Snapshot Testing?](#what-is-snapshot-testing)
  * [Property-Based Testing with Hypothesis](#property-based-testing-with-hypothesis)
* [4 · Integration Testing RAG Pipelines](#4-integration-testing-rag-pipelines)
  * [🤔 How Do You Test a RAG Pipeline?](#how-do-you-test-a-rag-pipeline)
* [Knowledge Check](#knowledge-check)
  * [Question 1: Mock vs Real API](#question-1-mock-vs-real-api)
  * [Question 2: Snapshot Testing](#question-2-snapshot-testing)
  * [Question 3: Contract Testing](#question-3-contract-testing)
  * [Question 4: Testing Pyramid](#question-4-testing-pyramid)
* [Practical Practice](#practical-practice)
  * [Exercise 1: Write a Mock Test](#exercise-1-write-a-mock-test)
  * [Exercise 2: Contract Design](#exercise-2-contract-design)
  * [Exercise 3: Invariance Test Cases](#exercise-3-invariance-test-cases)
* [🎯 Summary & Key Takeaways](#summary-key-takeaways)
  * [🧠 Key Rules](#key-rules)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Traditional software is deterministic (2+2=4). LLMs are probabilistic (2+2 might be 'Four', '4.0', or 'Here is a poem about four'). Seniors use specialized testing (mocks, evals, contracts) to tame LLM chaos.

**Mental Model:** Snapshot testing saves a 'known good' response and compares future prompts against it. Contract testing (Pydantic) ensures the LLM's output shape is physically valid before downstream code touches it.

**Common Junior Pitfall:** Making live API calls to OpenAI inside unit tests. This costs money, takes minutes to run, and fails randomly due to network timeout or model updates. Always mock network calls in unit tests.

---


In [ ]:
# Cell 1 — Install dependencies
!pip install -q pytest hypothesis pydantic

## 1 · Unit Testing AI Components

### 🤔 What CAN Be Unit Tested in AI?

| Component | Testable? | How |
|-----------|-----------|-----|
| Prompt templates | ✅ Yes | Verify variables are inserted correctly |
| Input validation | ✅ Yes | Check Pydantic models reject bad inputs |
| Output parsing | ✅ Yes | Parse known JSON/text outputs |
| Chunking logic | ✅ Yes | Verify text is split correctly |
| Embedding pipeline | ✅ Yes | Check dimensions, type, normalization |
| LLM raw output | ❌ Hard | Non-deterministic, use mocking |
| Model quality | ❌ Hard | Use evaluation frameworks (NB40) |

### The #1 Pattern: Mock the LLM

Instead of calling a real LLM in tests (slow + expensive), **mock it** with a predetermined response.

In [ ]:
# Cell 2 — Unit testing with LLM mocking
from unittest.mock import Mock, patch, MagicMock
import json

# ---- The Production Code ----
class SentimentAnalyzer:
    """A class that uses an LLM to analyze sentiment."""
    def __init__(self, llm_client):
        self.llm = llm_client
    
    def build_prompt(self, text):
        """Build the prompt. This is UNIT TESTABLE."""
        return f"""Analyze the sentiment of this text. Return JSON only.
Text: \"{text}\"
Output format: {{"sentiment": "positive|negative|neutral", "confidence": 0.0-1.0}}"""
    
    def parse_response(self, raw_response):
        """Parse LLM response. This is UNIT TESTABLE."""
        try:
            result = json.loads(raw_response)
            if result['sentiment'] not in ['positive', 'negative', 'neutral']:
                raise ValueError(f"Invalid sentiment: {result['sentiment']}")
            if not 0 <= result['confidence'] <= 1:
                raise ValueError(f"Invalid confidence: {result['confidence']}")
            return result
        except (json.JSONDecodeError, KeyError) as e:
            return {'sentiment': 'unknown', 'confidence': 0.0, 'error': str(e)}
    
    def analyze(self, text):
        """Full pipeline. Tested with MOCKED LLM."""
        prompt = self.build_prompt(text)
        raw = self.llm.complete(prompt)
        return self.parse_response(raw)

# ---- The Tests ----
def test_build_prompt():
    """Test 1: Prompt template is correct (no LLM needed)."""
    analyzer = SentimentAnalyzer(llm_client=None)
    prompt = analyzer.build_prompt('I love this product')
    assert 'I love this product' in prompt, 'Text not in prompt'
    assert 'JSON' in prompt, 'Format instruction missing'
    assert 'sentiment' in prompt, 'Expected field missing'
    return '✅ Prompt template test PASSED'

def test_parse_valid_response():
    """Test 2: Parser handles valid JSON (no LLM needed)."""
    analyzer = SentimentAnalyzer(llm_client=None)
    result = analyzer.parse_response('{"sentiment": "positive", "confidence": 0.95}')
    assert result['sentiment'] == 'positive'
    assert result['confidence'] == 0.95
    return '✅ Valid response parsing test PASSED'

def test_parse_invalid_json():
    """Test 3: Parser handles garbage output gracefully."""
    analyzer = SentimentAnalyzer(llm_client=None)
    result = analyzer.parse_response('This is not JSON at all')
    assert result['sentiment'] == 'unknown'
    assert 'error' in result
    return '✅ Invalid JSON handling test PASSED'

def test_full_pipeline_with_mock():
    """Test 4: Full pipeline with MOCKED LLM (fast + free)."""
    mock_llm = Mock()
    mock_llm.complete.return_value = '{"sentiment": "negative", "confidence": 0.87}'
    
    analyzer = SentimentAnalyzer(llm_client=mock_llm)
    result = analyzer.analyze('The product broke after one day')
    
    assert result['sentiment'] == 'negative'
    mock_llm.complete.assert_called_once()  # Verify LLM was called exactly once
    return '✅ Full pipeline (mocked LLM) test PASSED'

# Run all tests
print('🧪 Unit Tests for AI Application')
print('=' * 50)
for test_fn in [test_build_prompt, test_parse_valid_response, test_parse_invalid_json, test_full_pipeline_with_mock]:
    result = test_fn()
    print(f'  {result}')

print(f'\n💡 Key: Tests 1-3 test DETERMINISTIC parts (no LLM needed).')
print(f'   Test 4 mocks the LLM to test the full flow without API costs.')

---

## 2 · Contract Testing: Schema Validation

### 🤔 What is a Contract Test?

A contract test verifies that inputs and outputs match their expected schemas. If your API changes its response format, contract tests catch it BEFORE users see errors.

**Analogy:** A contract test is like checking that a letter has the right stamp and address format BEFORE mailing. The content might vary, but the format must be correct.

In [ ]:
# Cell 3 — Contract testing with Pydantic
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional

# Define the CONTRACT (schema)
class RAGRequest(BaseModel):
    query: str = Field(min_length=1, max_length=1000)
    top_k: int = Field(default=5, ge=1, le=20)
    filters: Optional[dict] = None

class RAGChunk(BaseModel):
    text: str
    source: str
    score: float = Field(ge=0.0, le=1.0)

class RAGResponse(BaseModel):
    answer: str = Field(min_length=1)
    chunks: List[RAGChunk]
    model: str
    tokens_used: int = Field(ge=0)

# Contract tests
def test_valid_request():
    req = RAGRequest(query='What is our Q4 revenue?', top_k=5)
    assert req.query == 'What is our Q4 revenue?'
    return '✅ Valid request passes'

def test_empty_query_rejected():
    try:
        RAGRequest(query='', top_k=5)
        return '❌ Empty query should be rejected'
    except ValidationError:
        return '✅ Empty query correctly rejected'

def test_invalid_top_k_rejected():
    try:
        RAGRequest(query='test', top_k=100)  # max is 20
        return '❌ Invalid top_k should be rejected'
    except ValidationError:
        return '✅ Invalid top_k correctly rejected'

def test_response_schema():
    resp = RAGResponse(
        answer='Q4 revenue was $58.1M',
        chunks=[RAGChunk(text='Revenue report...', source='Q4_report.pdf', score=0.95)],
        model='gpt-4o',
        tokens_used=150
    )
    assert resp.tokens_used >= 0
    return '✅ Response schema validates correctly'

print('📋 Contract Tests')
print('=' * 50)
for test in [test_valid_request, test_empty_query_rejected, test_invalid_top_k_rejected, test_response_schema]:
    print(f'  {test()}')

print(f'\n💡 Contract tests are FAST (no LLM calls) and catch schema changes early.')

---

## 3 · Snapshot Testing for Prompts

### 🤔 What is Snapshot Testing?

When you change a prompt template, you want to know if the change BREAKS existing behavior. Snapshot testing saves the output from a known-good run and compares future runs against it.

```
First run:      prompt_v1 + input → output_A → SAVE as "snapshot"
After change:   prompt_v2 + input → output_B → COMPARE to snapshot
                                              → DIFFERENT! Review the change.
```

In [ ]:
# Cell 4 — Prompt regression tracking
import hashlib, json

class PromptVersionTracker:
    """Track prompt changes and detect regressions."""
    
    def __init__(self):
        self.versions = []
        self.snapshots = {}  # prompt_hash -> expected outputs
    
    def register_prompt(self, name, template, version):
        prompt_hash = hashlib.md5(template.encode()).hexdigest()[:8]
        entry = {'name': name, 'version': version, 'hash': prompt_hash, 'template_preview': template[:80]}
        self.versions.append(entry)
        return prompt_hash
    
    def save_snapshot(self, prompt_hash, test_inputs, expected_outputs):
        self.snapshots[prompt_hash] = {
            'inputs': test_inputs,
            'outputs': expected_outputs,
        }
    
    def check_regression(self, prompt_hash, new_outputs):
        """Compare new outputs to saved snapshots."""
        if prompt_hash not in self.snapshots:
            return 'NEW — no previous snapshot to compare'
        
        old = self.snapshots[prompt_hash]['outputs']
        changes = []
        for i, (old_out, new_out) in enumerate(zip(old, new_outputs)):
            if old_out != new_out:
                changes.append(f'  Input {i}: "{old_out[:40]}..." → "{new_out[:40]}..."')
        
        if changes:
            return f'⚠️ REGRESSION DETECTED ({len(changes)} changes):\n' + '\n'.join(changes)
        return '✅ No regression — outputs match snapshot'

tracker = PromptVersionTracker()

# Version 1 of our summarization prompt
v1_template = 'Summarize the following text in one sentence: {text}'
v1_hash = tracker.register_prompt('summarizer', v1_template, 'v1')
tracker.save_snapshot(v1_hash, 
    test_inputs=['Long article about AI...', 'Report on revenue...'],
    expected_outputs=['AI is transforming industries.', 'Revenue grew 15% YoY.']
)

# Version 2 — someone changed the prompt
v2_template = 'Provide a brief summary (max 1 sentence) of: {text}'
v2_hash = tracker.register_prompt('summarizer', v2_template, 'v2')

# Check regression
print('📸 Prompt Snapshot Testing')
print('=' * 60)
print(f'\nv1 hash: {v1_hash}')
print(f'v2 hash: {v2_hash}')
print(f'Hashes match: {v1_hash == v2_hash}')

# Same hash → check snapshot
result = tracker.check_regression(v1_hash, ['AI is transforming industries.', 'Revenue grew 15% YoY.'])
print(f'\nv1 re-test: {result}')

# Different hash → new prompt detected
result2 = tracker.check_regression(v2_hash, ['AI changes everything.', 'Revenue up 15%.'])
print(f'v2 test: {result2}')

print(f'\n💡 When the prompt hash changes, you know the prompt was modified.')
print(f'   Run regression tests to ensure the change doesn\'t break existing behavior.')

---
### Property-Based Testing with Hypothesis

Instead of writing specific test cases, property-based testing **generates hundreds of random inputs** and checks that properties always hold.

| Traditional Tests | Property-Based Tests |
|-------------------|---------------------|
| You write specific inputs | Framework generates random inputs |
| 5-10 test cases | 100+ auto-generated cases |
| May miss edge cases | Actively searches for edge cases |
| Tests the expected path | Tests the unexpected |


In [ ]:
# Property-based testing with hypothesis
from hypothesis import given, settings, strategies as st

# Property: chunking should NEVER lose data
@given(text=st.text(min_size=1, max_size=5000), chunk_size=st.integers(min_value=1, max_value=500))
@settings(max_examples=50)
def test_chunking_preserves_content(text, chunk_size):
    words = text.split()
    if not words:
        return  # empty text is trivially handled
    chunks = [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]
    reconstructed = ' '.join(chunks)
    assert reconstructed == ' '.join(words), 'Data lost in chunking!'

# Property: parsed sentiment should always be one of the valid values
@given(sentiment=st.sampled_from(['positive', 'negative', 'neutral']),
       confidence=st.floats(min_value=0.0, max_value=1.0))
@settings(max_examples=50)
def test_parse_always_returns_valid_sentiment(sentiment, confidence):
    import json
    raw = json.dumps({'sentiment': sentiment, 'confidence': confidence})
    result = json.loads(raw)
    assert result['sentiment'] in ['positive', 'negative', 'neutral']
    assert 0 <= result['confidence'] <= 1

# Run the property tests
print('Property-Based Tests (Hypothesis)')
print('=' * 50)
try:
    test_chunking_preserves_content()
    print('  [PASS] Chunking preserves content (50 random inputs)')
except Exception as e:
    print(f'  [FAIL] Chunking: {e}')

try:
    test_parse_always_returns_valid_sentiment()
    print('  [PASS] Parse returns valid sentiment (50 random inputs)')
except Exception as e:
    print(f'  [FAIL] Parse: {e}')

print('\nHypothesis generated 100 random test cases automatically!')
print('This catches edge cases you would never think to write manually.')


---

## 4 · Integration Testing RAG Pipelines

### 🤔 How Do You Test a RAG Pipeline?

A RAG pipeline has multiple stages. Test EACH stage independently:

| Stage | Test | What You Check |
|-------|------|---------------|
| Chunking | Unit test | Chunk sizes, overlap, no data loss |
| Embedding | Unit test | Correct dimensions, normalized |
| Retrieval | Integration test | Top-K returns relevant docs |
| Generation | Integration test | Answer is grounded in context |
| End-to-end | E2E test | Full pipeline gives correct answer |

In [ ]:
# Cell 5 — Integration testing a RAG pipeline
import numpy as np

class MockRAGPipeline:
    """A testable RAG pipeline with clear component boundaries."""
    
    def chunk(self, text, chunk_size=100):
        words = text.split()
        return [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]
    
    def embed(self, text):
        np.random.seed(hash(text) % 2**32)
        vec = np.random.randn(384)
        return vec / np.linalg.norm(vec)  # L2 normalized
    
    def retrieve(self, query_vec, doc_vecs, top_k=3):
        scores = [np.dot(query_vec, dv) for dv in doc_vecs]
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]

rag = MockRAGPipeline()

# Test 1: Chunking preserves all content
def test_chunking_no_data_loss():
    original = 'word ' * 250  # 250 words
    chunks = rag.chunk(original.strip(), chunk_size=100)
    reconstructed = ' '.join(chunks)
    assert len(reconstructed.split()) == len(original.strip().split()), 'Data lost in chunking!'
    return f'✅ Chunking: {len(chunks)} chunks, no data loss'

# Test 2: Embeddings are normalized
def test_embedding_normalized():
    vec = rag.embed('test query')
    assert vec.shape == (384,), f'Wrong dimensions: {vec.shape}'
    norm = np.linalg.norm(vec)
    assert abs(norm - 1.0) < 1e-6, f'Not normalized: norm={norm}'
    return f'✅ Embedding: dim={vec.shape[0]}, L2 norm={norm:.6f}'

# Test 3: Same text produces same embedding (deterministic)
def test_embedding_deterministic():
    v1 = rag.embed('hello world')
    v2 = rag.embed('hello world')
    assert np.allclose(v1, v2), 'Same input → different embeddings!'
    return '✅ Embedding: deterministic (same input → same output)'

# Test 4: Retrieval returns correct number of results
def test_retrieval_top_k():
    query = rag.embed('revenue')
    docs = [rag.embed(f'doc {i}') for i in range(10)]
    results = rag.retrieve(query, docs, top_k=3)
    assert len(results) == 3, f'Expected 3 results, got {len(results)}'
    scores = [s for _, s in results]
    assert scores == sorted(scores, reverse=True), 'Results not sorted by score'
    return f'✅ Retrieval: top-3 returned, properly ranked'

print('🔗 Integration Tests for RAG Pipeline')
print('=' * 55)
for test in [test_chunking_no_data_loss, test_embedding_normalized, test_embedding_deterministic, test_retrieval_top_k]:
    print(f'  {test()}')

print(f'\n💡 Each stage is tested independently. If retrieval fails,')
print(f'   you know the bug is in retrieval, not chunking or embedding.')

---
## Knowledge Check

### Question 1: Mock vs Real API
Your CI pipeline runs 200 tests. 50 of them call the OpenAI API directly. Why is this problematic and how do you fix it?

<details>
<summary>Click to reveal answer</summary>

Problems: (1) **Cost** -- 50 API calls per CI run * multiple runs per day = expensive. (2) **Speed** -- each call takes 1-5 seconds = 50-250 seconds just for LLM calls. (3) **Flakiness** -- network issues, rate limits, or model updates cause random failures. Fix: Mock the LLM client in unit tests. Only use real API calls in a small number of nightly E2E tests.
</details>

### Question 2: Snapshot Testing
You changed a prompt template and the snapshot test flagged a regression. Is the new prompt necessarily worse?

<details>
<summary>Click to reveal answer</summary>

**Not necessarily.** The snapshot test only tells you the output CHANGED -- it doesn't judge quality. The new prompt might produce better results. Snapshot testing is a **change detector**, not a quality assessor. You must review the difference and decide if the new output is acceptable. If it is, update the snapshot.
</details>

### Question 3: Contract Testing
A frontend developer reports that the API suddenly returns `{"result": "positive"}` instead of `{"sentiment": "positive"}`. Which test would have caught this?

<details>
<summary>Click to reveal answer</summary>

A **contract test** using Pydantic would have caught this immediately. The `RAGResponse` model defines `sentiment` as a required field. If the API returns `result` instead, Pydantic's `ValidationError` would fire before the bad data reaches any downstream consumer. Contract tests run in milliseconds and should be part of every CI pipeline.
</details>

### Question 4: Testing Pyramid
Your team has 5 unit tests and 200 E2E tests. What's wrong with this ratio?

<details>
<summary>Click to reveal answer</summary>

The testing pyramid is **inverted**. You should have MANY unit tests (fast, cheap, catch most bugs), SOME integration tests, and FEW E2E tests (slow, expensive, catch integration issues). 200 E2E tests means: (1) Very slow CI pipeline, (2) High API costs, (3) Frequent flaky failures. Refactor: mock the LLM in most tests, convert to unit tests, keep ~10 E2E tests for critical paths.
</details>


---
## Practical Practice

### Exercise 1: Write a Mock Test
Write a unit test for a `TranslationService` class that has a method `translate(text, target_lang)` which calls an LLM. Mock the LLM to return a known translation and verify the output.

### Exercise 2: Contract Design
Design a Pydantic model for an embedding API response that must contain: a list of float vectors (each 384 dimensions), the model name used, and token count. Add appropriate Field validators.

### Exercise 3: Invariance Test Cases
Write 5 invariance test cases for a resume screening model. Each should specify the original text, the modified text, and whether the prediction should change.


In [ ]:
# ===== EXERCISE SOLUTIONS =====
from unittest.mock import Mock
from pydantic import BaseModel, Field
from typing import List

print('Ex 1: Mock Translation Test')
mock_llm = Mock()
mock_llm.complete.return_value = 'Hola mundo'

class TranslationService:
    def __init__(self, llm): self.llm = llm
    def translate(self, text, target):
        return self.llm.complete(f'Translate to {target}: {text}')

svc = TranslationService(mock_llm)
result = svc.translate('Hello world', 'Spanish')
assert result == 'Hola mundo'
mock_llm.complete.assert_called_once()
print('  PASSED: Translation with mocked LLM')

print('\nEx 2: Embedding API Contract')
class EmbeddingResponse(BaseModel):
    embeddings: List[List[float]] = Field(..., min_length=1)
    model: str = Field(..., min_length=1)
    tokens_used: int = Field(..., ge=0)

resp = EmbeddingResponse(
    embeddings=[[0.1]*384],
    model='text-embedding-ada-002',
    tokens_used=5
)
print(f'  Valid: model={resp.model}, dims={len(resp.embeddings[0])}')

print('\nEx 3: Resume Screening Invariance Tests')
tests = [
    ('Name: John Smith', 'Name: Jane Smith', 'same', 'Name should not affect score'),
    ('MIT Computer Science', 'State University CS', 'same', 'University prestige bias'),
    ('5 years experience', '15 years experience', 'different', 'Experience should matter'),
    ('Based in New York', 'Based in Lagos', 'same', 'Location should not bias'),
    ('Managed team of 10', 'Led team of 10', 'same', 'Synonyms should not change score'),
]
for orig, mod, expected, reason in tests:
    print(f'  "{orig}" -> "{mod}"  expect: {expected} ({reason})')


---

## 🎯 Summary & Key Takeaways

| Strategy | What It Tests | Speed | Cost |
|----------|--------------|-------|------|
| Unit tests with mocks | Individual functions, parsers | ⚡ ms | Free |
| Contract tests | Input/output schemas | ⚡ ms | Free |
| Snapshot tests | Prompt template changes | ⚡ ms | Free |
| Integration tests | Component interactions | 🔄 seconds | Low |
| E2E tests | Full pipeline correctness | 🐢 seconds | API costs |

### 🧠 Key Rules

1. **Mock the LLM** in unit tests — never call real APIs
2. **Test the deterministic parts** — prompts, parsers, validators, chunkers
3. **Snapshot test your prompts** — detect unintended changes
4. **Contract test schemas** — catch format changes before production
5. **Reserve E2E tests** for CI/CD gates with real LLM calls (use the cheapest model)

**Next →** `41_model_deployment.ipynb` — Layer 5 — Advanced Model Deployment: Multi-Model Serving, vLLM & Kubernetes Auto-Scaling